# ARC Prize 2026 - ARC-AGI-2: baseline program-search solver

CPU-only, offline, no external models. It searches a library of grid transformations, keeps programs that
reproduce **all** training pairs of a task, and uses the top two as `attempt_1` / `attempt_2`.
Tasks with no fitting program get a safe fallback (identity / most common output).

This is a **baseline** meant to give you a valid, working submission. Expect a low score.
Top 2025 Kaggle scores (about 24%) came from test-time-trained LLMs; see the last cell for how to extend this.

In [ ]:
import json, glob, os, itertools, time
from collections import Counter
import numpy as np

# ---- locate data (Kaggle mounts competition data under /kaggle/input) ----
roots = glob.glob('/kaggle/input/*arc*') or ['/kaggle/input']
def find(pattern):
    for r in roots:
        hits = glob.glob(os.path.join(r, '**', pattern), recursive=True)
        if hits: return hits[0]
    return None

test_path = find('*test_challenges.json')
train_path = find('*training_challenges.json')
train_sol_path = find('*training_solutions.json')
eval_path = find('*evaluation_challenges.json')
eval_sol_path = find('*evaluation_solutions.json')
print('test:', test_path, '\ntrain:', train_path, '\neval:', eval_path)
load = lambda p: json.load(open(p)) if p else {}

In [ ]:
A = np.array
def g(x): return A(x, dtype=int)

# ---------- helpers ----------
def bg_color(a):
    return Counter(a.ravel().tolist()).most_common(1)[0][0]

def components(a, bg, diag=True, by_color=True):
    """Connected components of non-background cells. Returns list of (mask, color-set)."""
    h, w = a.shape; seen = np.zeros_like(a, bool); out = []
    nbrs = [(1,0),(-1,0),(0,1),(0,-1)] + ([(1,1),(1,-1),(-1,1),(-1,-1)] if diag else [])
    for i in range(h):
        for j in range(w):
            if a[i,j] == bg or seen[i,j]: continue
            col = a[i,j]; stack = [(i,j)]; seen[i,j] = True; cells_ = []
            while stack:
                y, x = stack.pop(); cells_.append((y,x))
                for dy, dx in nbrs:
                    ny, nx = y+dy, x+dx
                    if 0<=ny<h and 0<=nx<w and not seen[ny,nx] and a[ny,nx]!=bg and (not by_color or a[ny,nx]==col):
                        seen[ny,nx] = True; stack.append((ny,nx))
            m = np.zeros_like(a, bool)
            for y, x in cells_: m[y,x] = True
            out.append(m)
    return out

def bbox_crop(a, m):
    ys, xs = np.where(m)
    return a[ys.min():ys.max()+1, xs.min():xs.max()+1]

# ---------- grid -> grid primitives (no parameters learned) ----------
GEOM = {
 'id': lambda a: a, 'rot90': lambda a: np.rot90(a,1), 'rot180': lambda a: np.rot90(a,2), 'rot270': lambda a: np.rot90(a,3),
 'flipud': np.flipud, 'fliplr': np.fliplr, 'transpose': lambda a: a.T, 'antitranspose': lambda a: np.rot90(a,2).T,
}
def crop_nonbg(a):
    bg = bg_color(a); m = a != bg
    return bbox_crop(a, m) if m.any() else a
def largest_obj(a):
    bg = bg_color(a); cs = components(a, bg)
    return bbox_crop(a, max(cs, key=lambda m: m.sum())) if cs else a
def smallest_obj(a):
    bg = bg_color(a); cs = components(a, bg)
    return bbox_crop(a, min(cs, key=lambda m: m.sum())) if cs else a
def largest_obj_multi(a):
    bg = bg_color(a); cs = components(a, bg, by_color=False)
    return bbox_crop(a, max(cs, key=lambda m: m.sum())) if cs else a
def most_colorful_obj(a):
    bg = bg_color(a); cs = components(a, bg, by_color=False)
    return bbox_crop(a, max(cs, key=lambda m: len(set(a[m].tolist())))) if cs else a
def mirror_h(a): return np.hstack([a, np.fliplr(a)])
def mirror_v(a): return np.vstack([a, np.flipud(a)])
def mirror_4(a): return mirror_v(mirror_h(a))
def top_half(a): return a[:a.shape[0]//2]
def bottom_half(a): return a[(a.shape[0]+1)//2:]
def left_half(a): return a[:, :a.shape[1]//2]
def right_half(a): return a[:, (a.shape[1]+1)//2:]
def gravity_down(a):
    bg = bg_color(a); out = np.full_like(a, bg)
    for j in range(a.shape[1]):
        col = [v for v in a[:,j] if v != bg]
        if col: out[-len(col):, j] = col
    return out
def gravity_up(a): return np.flipud(gravity_down(np.flipud(a)))
def gravity_right(a): return gravity_down(a.T).T
def gravity_left(a): return gravity_up(a.T).T
def dedup_rows_cols(a):
    rows = [0]+[i for i in range(1,a.shape[0]) if not np.array_equal(a[i],a[i-1])]
    a = a[rows]
    cols = [0]+[j for j in range(1,a.shape[1]) if not np.array_equal(a[:,j],a[:,j-1])]
    return a[:, cols]
def symmetrize(a):
    """Fill cells of the most common 'hole' color using mirror symmetries."""
    out = a.copy(); cnt = Counter(a.ravel().tolist())
    for hole in [c for c,_ in cnt.most_common()][::-1][:3]:
        o = a.copy()
        for t in (np.fliplr, np.flipud, lambda x: x.T if x.shape[0]==x.shape[1] else x, lambda x: np.rot90(x,2)):
            b = t(a); m = (o == hole) & (b != hole); o[m] = b[m]
        if (o != a).any(): return o
    return out
def fill_enclosed(a):
    bg = bg_color(a); h, w = a.shape; reach = np.zeros_like(a, bool); st = []
    for i in range(h):
        for j in range(w):
            if (i in (0,h-1) or j in (0,w-1)) and a[i,j]==bg: reach[i,j]=True; st.append((i,j))
    while st:
        y,x = st.pop()
        for dy,dx in ((1,0),(-1,0),(0,1),(0,-1)):
            ny,nx=y+dy,x+dx
            if 0<=ny<h and 0<=nx<w and not reach[ny,nx] and a[ny,nx]==bg: reach[ny,nx]=True; st.append((ny,nx))
    out = a.copy(); out[(a==bg)&~reach] = -1   # -1 = to be colour-mapped later
    return out
def tile2(a): return np.tile(a,(2,2))
def tile3(a): return np.tile(a,(3,3))
def up2(a): return np.kron(a, np.ones((2,2),int))
def up3(a): return np.kron(a, np.ones((3,3),int))
def down2(a): return a[::2, ::2]
def self_kron(a):
    bg = bg_color(a); out = np.full((a.shape[0]**2, a.shape[1]**2), bg)
    for i in range(a.shape[0]):
        for j in range(a.shape[1]):
            if a[i,j] != bg: out[i*a.shape[0]:(i+1)*a.shape[0], j*a.shape[1]:(j+1)*a.shape[1]] = a
    return out
def split_and(a, op):
    """Split grid in two halves (optionally separated by a line) and combine cellwise."""
    h, w = a.shape; cands = []
    if w % 2 == 1: cands.append((a[:, :w//2], a[:, w//2+1:]))
    if w % 2 == 0: cands.append((a[:, :w//2], a[:, w//2:]))
    if h % 2 == 1: cands.append((a[:h//2], a[h//2+1:]))
    if h % 2 == 0: cands.append((a[:h//2], a[h//2:]))
    return cands
def make_split(op):
    def f(a):
        for x, y in split_and(a, op):
            if x.shape == y.shape:
                return op(x, y)
        return a
    return f
OPS = {
 'and': lambda x,y: ((x!=0)&(y!=0)).astype(int),
 'or':  lambda x,y: ((x!=0)|(y!=0)).astype(int),
 'xor': lambda x,y: ((x!=0)^(y!=0)).astype(int),
 'nor': lambda x,y: ((x==0)&(y==0)).astype(int),
 'left_minus_right': lambda x,y: ((x!=0)&(y==0)).astype(int),
 'overlay': lambda x,y: np.where(y!=0,y,x),
}

PRIMS = dict(GEOM)
PRIMS.update({
 'crop_nonbg': crop_nonbg, 'largest_obj': largest_obj, 'smallest_obj': smallest_obj,
 'largest_obj_multi': largest_obj_multi, 'most_colorful_obj': most_colorful_obj,
 'mirror_h': mirror_h, 'mirror_v': mirror_v, 'mirror_4': mirror_4,
 'top_half': top_half, 'bottom_half': bottom_half, 'left_half': left_half, 'right_half': right_half,
 'gravity_down': gravity_down, 'gravity_up': gravity_up, 'gravity_left': gravity_left, 'gravity_right': gravity_right,
 'dedup': dedup_rows_cols, 'symmetrize': symmetrize, 'fill_enclosed': fill_enclosed,
 'tile2': tile2, 'tile3': tile3, 'up2': up2, 'up3': up3, 'down2': down2, 'self_kron': self_kron,
})
for n, op in OPS.items(): PRIMS['split_'+n] = make_split(op)
print(len(PRIMS), 'primitives')

In [ ]:
# ---------- colour-map learning: after a program, learn a global colour substitution ----------
def learn_cmap(pairs):
    """pairs: list of (predicted, target). Returns dict or None if inconsistent / shape mismatch."""
    m = {}
    for p, t in pairs:
        if p.shape != t.shape: return None
        for a_, b_ in zip(p.ravel().tolist(), t.ravel().tolist()):
            if m.setdefault(a_, b_) != b_: return None
    return m
def apply_cmap(a, m): 
    out = a.copy()
    for k, v in m.items(): out[a == k] = v
    return out

def safe(f, a):
    try:
        r = f(a)
        if r.ndim != 2 or r.size == 0 or r.shape[0] > 30 or r.shape[1] > 30: return None
        return r
    except Exception:
        return None

def solve_task(train, test_inputs, max_depth=2, time_budget=20):
    """Return list (per test input) of up to 2 candidate output grids."""
    ins = [g(p['input']) for p in train]; outs = [g(p['output']) for p in train]
    tins = [g(x) for x in test_inputs]
    names = list(PRIMS); t0 = time.time(); found = []   # (complexity, chain, cmap)
    def check(chain):
        preds = []
        for a in ins:
            r = a
            for n in chain:
                r = safe(PRIMS[n], r)
                if r is None: return None
            preds.append(r)
        if all(p.shape == o.shape and (p == o).all() for p, o in zip(preds, outs)): return {}
        if all(p.shape == o.shape for p, o in zip(preds, outs)):
            return learn_cmap(list(zip(preds, outs)))   # may be None
        return None
    for depth in range(1, max_depth+1):
        for chain in itertools.product(names, repeat=depth):
            if time.time() - t0 > time_budget: break
            if depth == 2 and chain[0] == 'id': continue
            cm = check(chain)
            if cm is not None: found.append((depth + (0.5 if cm else 0), chain, cm))
        if len(found) >= 2 or time.time() - t0 > time_budget: break
    found.sort(key=lambda x: x[0])
    results = [[] for _ in tins]
    for _, chain, cm in found:
        for k, a in enumerate(tins):
            r = a
            for n in chain:
                r = safe(PRIMS[n], r)
                if r is None: break
            if r is None: continue
            if cm: r = apply_cmap(r, cm)
            if (r < 0).any(): r = np.where(r < 0, 0, r)
            if not any(np.array_equal(r, q) for q in results[k]): results[k].append(r)
    return results

def fallback(train, a):
    """Cheap guesses used when no program fits."""
    outs = [g(p['output']) for p in train]; ins = [g(p['input']) for p in train]
    guesses = []
    if all(i.shape == o.shape for i, o in zip(ins, outs)): guesses.append(g(a))      # identity
    if len({o.shape for o in outs}) == 1 and all(np.array_equal(outs[0], o) for o in outs): guesses.append(outs[0])
    guesses.append(g(a))
    return guesses

In [ ]:
# ---------- local sanity check on training/evaluation tasks (skipped if files absent) ----------
def evaluate(chal, sols, limit=100):
    ok = tot = 0; t0 = time.time()
    for tid in list(chal)[:limit]:
        task = chal[tid]; res = solve_task(task['train'], [t['input'] for t in task['test']], time_budget=5)
        for k, sol in enumerate(sols[tid]):
            cands = res[k][:2] or []
            tot += 1; ok += any(np.array_equal(c, g(sol)) for c in cands)
    print(f'solved {ok}/{tot} test grids in {time.time()-t0:.0f}s')
for cp, sp in ((train_path, train_sol_path), (eval_path, eval_sol_path)):
    if cp and sp: print(os.path.basename(cp)); evaluate(load(cp), load(sp), limit=60)

In [ ]:
# ---------- build submission.json ----------
test = load(test_path)
submission = {}; t0 = time.time(); n_solved = 0
for idx, (tid, task) in enumerate(test.items()):
    tins = [t['input'] for t in task['test']]
    try: res = solve_task(task['train'], tins, time_budget=15)
    except Exception: res = [[] for _ in tins]
    entries = []
    for k, a in enumerate(tins):
        cands = list(res[k])
        if cands: n_solved += 1
        for fb in fallback(task['train'], a):
            if len(cands) >= 2: break
            if not any(np.array_equal(fb, c) for c in cands): cands.append(fb)
        while len(cands) < 2: cands.append(g(a))
        entries.append({'attempt_1': cands[0].tolist(), 'attempt_2': cands[1].tolist()})
    submission[tid] = entries
    if idx % 20 == 0: print(idx, len(test), f'{time.time()-t0:.0f}s')
json.dump(submission, open('submission.json', 'w'))
print('tasks:', len(submission), '| test grids with a fitted program:', n_solved)

## Next steps (what actually moves the score)
Kaggle limits: **no internet**, fixed GPU/CPU time per run. Public standings use a semi-private set; prizes use the private set.

1. **Test-time training (TTT)**: fine-tune a small open LLM (weights uploaded as a Kaggle Dataset) on the augmented
   train pairs of each test task at inference time, then decode with augmentation voting. This is the family of approaches
   behind the top 2025 scores.
2. **Pretrain on synthetic ARC-like data** (re-arc, ConceptARC, public eval set) in a separate GPU notebook; save weights as a Dataset.
3. **Keep this DSL solver as a first pass** and let the LLM handle tasks with no fitting program.
4. Check the output format against the competition's `sample_submission.json` before submitting.